LAB 7

Paraa cada consulta se calculan Precisión, Recuerdo, Medida F y Precisión R tomando los primeros Z documentos del ranking.

Entrada (práctica 2): qrels.text, los juicios de relevancia de CACM (rlv-ass), con líneas `consulta documento 0 0`.

Entrada (ranking del sistema): CACM_tf_idf_rels.txt, con líneas `consulta documento coseno` ordenadas de mayor a menor coseno.


In [1]:
import os

In [2]:
import numpy as np

In [ ]:

ruta_juicios = r"C:\Users\Alex\Desktop\RI\Lab2\qrels.text"
ruta_ranking = r"C:\Users\Alex\Desktop\RI\lab5\CACM_tf_idf_rels.txt"
ruta_salida = r"C:\Users\Alex\Desktop\RI\Lab7\CACM_metricas_Z100.txt"



Cada línea de qrels.text indica que un documento es relevante para una consulta. Se guarda un conjunto de documentos
relevantes por consulta. Los números se convierten a entero para que "01" y "001" se refieran a la misma consulta.

In [4]:
# devuelve {consulta: {documentos relevantes}}
def leer_juicios(ruta):
    relevantes = {}
    with open(ruta, encoding="utf-8") as f:
        for linea in f:
            campos = linea.split()
            if len(campos) >= 2:
                relevantes.setdefault(int(campos[0]), set()).add(int(campos[1]))
    return relevantes

In [5]:
relevantes = leer_juicios(ruta_juicios)
print("Consultas con juicios de relevancia:", len(relevantes))
print("Total de pares consulta-documento relevantes:", sum(len(v) for v in relevantes.values()))
print("Documentos relevantes de la consulta 1:", sorted(relevantes[1]))

Consultas con juicios de relevancia: 52
Total de pares consulta-documento relevantes: 796
Documentos relevantes de la consulta 1: [1410, 1572, 1605, 2020, 2358]



El archivo de salida del sistema ya trae los documentos de cada consulta ordenados de mayor a menor similitud,
así que basta con conservar ese orden en una lista por consulta.

In [6]:
# devuelve {consulta: [documentos en orden de ranking]}
def leer_ranking(ruta):
    ranking = {}
    with open(ruta, encoding="utf-8") as f:
        for linea in f:
            campos = linea.split()
            if len(campos) == 3:
                ranking.setdefault(int(campos[0]), []).append(int(campos[1]))
    return ranking

In [7]:
ranking = leer_ranking(ruta_ranking)
print("Consultas en el ranking:", len(ranking))
print("Primeros 10 documentos de la consulta 1:", ranking[1][:10])

Consultas en el ranking: 64
Primeros 10 documentos de la consulta 1: [1938, 31, 1071, 379, 1572, 971, 2218, 163, 1657, 61]


VALIDACIONES

In [8]:
consultas_evaluadas = sorted(q for q in relevantes if q in ranking)
sin_juicios = sorted(q for q in ranking if q not in relevantes)
print("Consultas que se evaluan:", len(consultas_evaluadas))
print("Consultas sin juicios de relevancia (no se evaluan):", sin_juicios)
print("Consultas con juicios pero sin ranking:", sorted(q for q in relevantes if q not in ranking))
print("Rankings sin documentos repetidos:", all(len(v) == len(set(v)) for v in ranking.values()))

Consultas que se evaluan: 52
Consultas sin juicios de relevancia (no se evaluan): [34, 35, 41, 46, 47, 50, 51, 52, 53, 54, 55, 56]
Consultas con juicios pero sin ranking: []
Rankings sin documentos repetidos: True


MÉTRICAS

Para una consulta con conjunto de relevantes R y los primeros Z documentos del ranking:

- Precisión = relevantes recuperados / documentos recuperados. Si el sistema devolvió menos de Z documentos, se divide entre los que devolvió.
- Recuerdo = relevantes recuperados / total de relevantes de la consulta.
- Medida F = 2 · P · R / (P + R), y vale 0 cuando P y R son 0.
- Precisión R = precisión en la posición |R|, donde |R| es el número de documentos relevantes de la consulta. Si |R| es mayor que Z, solo se revisan los primeros Z documentos, pero se sigue dividiendo entre |R|.

In [ ]:
# metricas de una consulta tomando los primeros Z documentos del ranking
def metricas(lista, relevantes_q, Z):
    recuperados = lista[:Z]
    w = sum(1 for d in recuperados if d in relevantes_q)        
    precision = w / len(recuperados) if recuperados else 0.0
    recuerdo = w / len(relevantes_q)
    medida_f = 2 * precision * recuerdo / (precision + recuerdo) if precision + recuerdo > 0 else 0.0
    r = len(relevantes_q)
    precision_r = sum(1 for d in lista[:min(r, Z)] if d in relevantes_q) / r
    return precision, recuerdo, medida_f, precision_r

In [10]:
# devuelve {consulta: (precision, recuerdo, medida_f, precision_r)}
def evaluar(Z):
    return {q: metricas(ranking[q], relevantes[q], Z) for q in consultas_evaluadas}

def mostrar(resultados, Z):
    print("Z =", Z)
    print("%-9s %-10s %-10s %-10s %-10s" % ("Consulta", "Precision", "Recuerdo", "Medida F", "Precision R"))
    for q, (p, r, f, pr) in resultados.items():
        print("%-9d %-10.4f %-10.4f %-10.4f %-10.4f" % (q, p, r, f, pr))
    promedios = np.mean(list(resultados.values()), axis=0)
    print("%-9s %-10.4f %-10.4f %-10.4f %-10.4f" % ("Promedio", *promedios))


Z indica cuántos documentos del ranking se toman para calcular las métricas.

In [11]:
try:
    Z = int(input("Numero de documentos Z: "))
except (NotImplementedError, EOFError, ValueError):
    Z = 100
    print("No se pudo leer Z de pantalla, se usa Z =", Z)
if Z <= 0:
    raise ValueError("Z debe ser un entero positivo")

No se pudo leer Z de pantalla, se usa Z = 100


In [ ]:
mostrar(evaluar(Z), Z)

Z = 100
Consulta  Precision  Recuerdo   Medida F   Precision R
1         0.0400     0.8000     0.0762     0.2000    
2         0.0000     0.0000     0.0000     0.0000    
3         0.0200     0.3333     0.0377     0.1667    
4         0.0400     0.3333     0.0714     0.1667    
5         0.0700     0.8750     0.1296     0.1250    
6         0.0300     1.0000     0.0583     0.0000    
7         0.1200     0.4286     0.1875     0.3214    
8         0.0300     1.0000     0.0583     0.0000    
9         0.0700     0.7778     0.1284     0.1111    
10        0.2500     0.7143     0.3704     0.6286    
11        0.1300     0.6842     0.2185     0.4211    
12        0.0300     0.6000     0.0571     0.4000    
13        0.0800     0.7273     0.1441     0.3636    
14        0.3300     0.7500     0.4583     0.4773    
15        0.0800     0.8000     0.1455     0.2000    
16        0.0600     0.3529     0.1026     0.1176    
17        0.0900     0.5625     0.1552     0.1875    
18        0.0700   



Sin importar el valor leído de pantalla, el archivo se genera con Z = 100

In [13]:
def guardar_resultados(resultados, ruta, decimales=4):
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as salida:
        salida.write("Consulta\tPrecision\tRecuerdo\tMedidaF\tPrecisionR\n")
        for q, valores in resultados.items():
            salida.write(str(q) + "\t" + "\t".join(f"{v:.{decimales}f}" for v in valores) + "\n")
    print("Archivo generado:", ruta)

In [14]:
resultados_100 = evaluar(100)
guardar_resultados(resultados_100, ruta_salida)

Archivo generado: C:\Users\Alex\Desktop\RI\Lab7\CACM_metricas_Z100.txt


Veriicamos

In [15]:
with open(ruta_salida, encoding="utf-8") as f:
    lineas = [linea.split("\t") for linea in f.read().splitlines()]
for campos in lineas[:6]:
    print("\t".join(campos))
print("...")
print("Lineas de datos:", len(lineas) - 1, "| consultas evaluadas:", len(consultas_evaluadas))
print("Todas las lineas tienen 5 campos:", all(len(c) == 5 for c in lineas))
print("Todos los valores entre 0 y 1:", all(0 <= float(v) <= 1 for c in lineas[1:] for v in c[1:]))

Consulta	Precision	Recuerdo	MedidaF	PrecisionR
1	0.0400	0.8000	0.0762	0.2000
2	0.0000	0.0000	0.0000	0.0000
3	0.0200	0.3333	0.0377	0.1667
4	0.0400	0.3333	0.0714	0.1667
5	0.0700	0.8750	0.1296	0.1250
...
Lineas de datos: 52 | consultas evaluadas: 52
Todas las lineas tienen 5 campos: True
Todos los valores entre 0 y 1: True


In [16]:
q = 1
lista, rel_q = ranking[q], relevantes[q]
posiciones = [i + 1 for i, d in enumerate(lista) if d in rel_q]
print("Consulta", q, "| relevantes:", len(rel_q), "| documentos en el ranking:", len(lista))
print("Posiciones de los relevantes en el ranking:", posiciones)

w = sum(1 for p in posiciones if p <= 100)
p_man = w / min(100, len(lista))
r_man = w / len(rel_q)
f_man = 2 * p_man * r_man / (p_man + r_man) if p_man + r_man > 0 else 0.0
pr_man = sum(1 for p in posiciones if p <= min(len(rel_q), 100)) / len(rel_q)
print("Relevantes en los primeros 100:", w)
print("A mano:       P = %.4f  R = %.4f  F = %.4f  PR = %.4f" % (p_man, r_man, f_man, pr_man))
print("Del programa: P = %.4f  R = %.4f  F = %.4f  PR = %.4f" % resultados_100[q])
print("Coinciden:", np.allclose([p_man, r_man, f_man, pr_man], resultados_100[q]))

Consulta 1 | relevantes: 5 | documentos en el ranking: 798
Posiciones de los relevantes en el ranking: [5, 23, 30, 49, 653]
Relevantes en los primeros 100: 4
A mano:       P = 0.0400  R = 0.8000  F = 0.0762  PR = 0.2000
Del programa: P = 0.0400  R = 0.8000  F = 0.0762  PR = 0.2000
Coinciden: True


COMPORTAMIENTO AL VARIAR Z

Al aumentar Z el recuerdo nunca disminuye, porque solo se agregan documentos y la precisión normalmente baja,
porque los documentos que se agregan son cada vez menos parecidos a la consulta.

In [17]:
print("%-6s %-10s %-10s %-10s %-10s" % ("Z", "Precision", "Recuerdo", "Medida F", "Precision R"))
for z in [5, 10, 20, 50, 100, 200, 500]:
    print("%-6d %-10.4f %-10.4f %-10.4f %-10.4f" % (z, *np.mean(list(evaluar(z).values()), axis=0)))

Z      Precision  Recuerdo   Medida F   Precision R
5      0.3462     0.1664     0.1821     0.1440    
10     0.2750     0.2512     0.2096     0.1884    
20     0.2173     0.3396     0.2211     0.2357    
50     0.1435     0.5386     0.2011     0.2618    
100    0.0898     0.6454     0.1465     0.2618    
200    0.0554     0.7363     0.0987     0.2618    
500    0.0289     0.8151     0.0543     0.2618    
